## Setup

In [1]:
import optuna
from optuna.samplers import TPESampler
from optuna.trial import Trial
from optuna.study import create_study
import diff_cont
import lambda_cont
import libs.agent_infra as ai
import os

import json
import datetime
import itertools
import constants as Cs
import concurrent.futures
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from scipy.spatial.distance import pdist


SEEDS = [101, 102, 103]
TEST_EVAL_EPS = 6

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-12 01:05:03.773792: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-12 01:05:03.817211: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-12 01:05:04.892883: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF

## Add Novelty Diff

In [ ]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
    decay = trial.suggest_float("decay", 0.5, 5, step=0.5)

    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env="lunarlander",
                container="add_novelty",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                fitness_weight=fit_w,
                decay=decay,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=25)
study = create_study(direction="maximize", sampler=sampler, study_name="diff_add_novelty_exp", storage="sqlite:///Data/optuna/lunarlander/add_novelty/diff.db", load_if_exists=True)
study.optimize(objective, n_trials=180, n_jobs=5)
print(study.best_trials)

## Sub Novelty

In [4]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
    decay = trial.suggest_float("decay", 0.5, 5, step=0.5)

    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env="lunarlander",
                container="sub_novelty",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                fitness_weight=fit_w,
                decay=decay,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    behaviors = list(map(lambda x:x[1], fitnesses))
    fitnesses = list(map(lambda x:x[0], fitnesses))
    diversity = pdist(np.array(behaviors)).mean()
    assert diversity > 0
    return np.max(fitnesses) + diversity


sampler = TPESampler(n_startup_trials=15)
study = create_study(direction="maximize", sampler=sampler, study_name="diff_sub_novelty_exp", storage="sqlite:///Data/optuna/lunarlander/sub_novelty/diff.db", load_if_exists=True)
study.optimize(objective, n_trials=170, n_jobs=5)
print(study.best_trials)

[I 2026-06-12 07:09:33,485] Using an existing study with name 'diff_sub_novelty_exp' instead of creating a new one.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of

   	                        fitness                        	                            novelty                             
   	-------------------------------------------------------	----------------------------------------------------------------
gen	avg     	gen	max    	min     	std    	avg     	gen	max     	min      	std     
0  	-398.368	0  	-95.519	-674.767	177.273	0.200258	0  	0.772351	0.0198629	0.155315
   	                       fitness                        	                        novelty                         
   	------------------------------------------------------	--------------------------------------------------------
gen	avg     	gen	max     	min    	std   	avg     	gen	max     	min    	std     
0  	-374.247	0  	-123.035	-830.16	166.66	0.313233	0  	0.858761	0.10562	0.184257
   	                        fitness                        	                            novelty                             
   	-------------------------------------------------------	-------

[I 2026-06-12 07:49:26,540] Trial 9 finished with value: -115.00881920831831 and parameters: {'lambda': 50, 'mutation_rate': 0.18, 'cross_rate': 0.5, 'start_fit_w': 0.30000000000000004, 'decay': 4.0}. Best is trial 9 with value: -115.00881920831831.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: Runt

   	                        fitness                        	                    novelty                     
   	-------------------------------------------------------	------------------------------------------------
gen	avg     	gen	max     	min    	std    	avg     	gen	max	min	std     
0  	-403.441	0  	-112.317	-665.77	145.146	0.473986	0  	1  	0  	0.262255
   	                           fitness                            	                novelty                 
   	--------------------------------------------------------------	----------------------------------------
gen	avg     	gen	max     	min     	std   	avg    	gen	max	min	std     
0  	-421.528	0  	-100.339	-861.524	163.09	0.57804	0  	1  	0  	0.214257
   	                        fitness                        	                    novelty                     
   	-------------------------------------------------------	------------------------------------------------
gen	avg     	gen	max     	min    	std    	avg     	gen	max	min

[I 2026-06-12 07:54:16,638] Trial 7 finished with value: -24.16629413811712 and parameters: {'lambda': 60, 'mutation_rate': 0.29, 'cross_rate': 0.7, 'start_fit_w': 0.9000000000000001, 'decay': 1.0}. Best is trial 7 with value: -24.16629413811712.
[I 2026-06-12 07:54:16,642] Trial 5 finished with value: -110.98694008548178 and parameters: {'lambda': 60, 'mutation_rate': 0.49, 'cross_rate': 0.5, 'start_fit_w': 0.2, 'decay': 5.0}. Best is trial 7 with value: -24.16629413811712.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. C

...

[I 2026-06-13 02:09:11,265] Trial 173 finished with value: 46.06928466262888 and parameters: {'lambda': 40, 'mutation_rate': 0.19, 'cross_rate': 0.4, 'start_fit_w': 0.7, 'decay': 1.5}. Best is trial 110 with value: 86.2781801840644.


3  	-362.421	3  	-82.4602	-567.108	143.209	0.347976	3  	0.700626	0.0528531 	0.181675
3  	-391.145	3  	-49.1667	-710.587	183.908	0.374386	3  	0.709259	0.0314537 	0.18584 
4  	-414.784	4  	-89.2139	-712.807	163.567	0.349032	4  	0.705818	0.00226566	0.18626 
3  	-433.034	3  	-5.08742	-682.89 	193.866	0.313948	3  	0.702721	0.0521803	0.171245
4  	-338.096	4  	-62.6076	-580.358	152.258	0.375319	4  	0.702719	0.0175249 	0.168312
5  	-372.534	5  	-56.6589	-712.513	210.318	0.399193	5  	0.77244 	0.00369948	0.224055
5  	-360.981	5  	-89.4042	-599.591	154.592	0.36887 	5  	0.709362	0.0621061 	0.194274
4  	-447.308	4  	-33.8793	-682.89 	201.012	0.331475	4  	0.70208 	0.0989744	0.17063 
6  	-386.298	6  	-53.111 	-712.513	174.245	0.395532	6  	0.702306	0.00173568	0.175382
6  	-363.026	6  	-102.085	-616.413	150.954	0.378306	6  	0.704122	0.00101562	0.185787
7  	-450.084	7  	-96.0497	-751.6  	192.203	0.346783	7  	0.701043	0.00119935	0.204586
5  	-445.348	5  	-117.028	-724.177	192.361	0.375008	5  	0.708292	0.

[I 2026-06-13 02:20:27,657] Trial 174 finished with value: 46.06928466262888 and parameters: {'lambda': 40, 'mutation_rate': 0.19, 'cross_rate': 0.4, 'start_fit_w': 0.7, 'decay': 1.5}. Best is trial 110 with value: 86.2781801840644.
Process ForkProcess-41:
Process ForkProcess-32:
Process ForkProcess-31:
Process ForkProcess-42:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/ho

KeyboardInterrupt: 

[W 2026-06-13 05:23:21,466] Trial 6 failed with parameters: {'lambda': 70, 'mutation_rate': 0.11, 'cross_rate': 0.4, 'start_fit_w': 0.4, 'decay': 1.5} because of the following error: BrokenProcessPool('A process in the process pool was terminated abruptly while the future was running or pending.').
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_158192/1002431591.py", line 23, in objective
    pops = [f.result()[1] for f in futures]
  File "/tmp/ipykernel_158192/1002431591.py", line 23, in <listcomp>
    pops = [f.result()[1] for f in futures]
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/concurrent/futures/_base.py", line 458, in result
    return self.__get_result()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/concurrent/futures/_base.py", line 403, in __get_result
    raise self

## Fit_Archiving

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="fit_archiving",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
diff_fita_study = create_study(direction="maximize", study_name="diff_fit_archiving_reloaded", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=150, n_jobs=5)
print(diff_fita_study.best_trials)

## Novlety Limit Archiving

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    limit = trial.suggest_float("limit", -200, -50, step=10)

    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="novelty_limit",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                limit=limit,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
diff_fita_study = create_study(direction="maximize", study_name="diff_novelty_limit", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/novelty_limit/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=170, n_jobs=5)
print(diff_fita_study.best_trials)

## Novelty Archiving

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="novelty_archiving",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
diff_fita_study = create_study(direction="maximize", study_name="diff_novelty_archiving", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/novelty_archiving/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=150, n_jobs=5)
print(diff_fita_study.best_trials)